Force errors of MACE uni and Trial 0 against Runs2

In [2]:

import numpy as np
import pandas as pd
import torch
from ase.io import read
from mace.calculators import MACECalculator

datasets = [
    "/home/mehuldarak/athena/DFTFE_labelled_data/completedRuns_2.extxyz"
]

model_path = "/home/mehuldarak/MACE_models/universal_09072025/mace-omat-0-medium.model"

torch.set_default_dtype(torch.float32)

calc = MACECalculator(
    model_path=model_path,
    device="cuda",
    default_dtype="float32"
)

rows = []

for file in datasets:
    structures = read(file, ":")

    for atoms in structures:
        atoms.calc = calc

        pred_forces = atoms.get_forces()
        ref_forces = atoms.arrays["REF_forces"]

        # absolute force error
        diff = pred_forces - ref_forces
        atom_errors = np.linalg.norm(diff, axis=1)

        # one scalar per structure
        force_error = atom_errors.mean()

        rows.append({
            "source_file": atoms.info["filename"],
            "force_error": 1000*force_error
        })

df = pd.DataFrame(rows)

print(df.head())

df.to_csv("maceUni_vs_runs2_force_errors.csv", index=False)

/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/mace/calculators/mace.py:166: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head default out of ['default']
Default dtype float32 does not match model dtype float64, converting models to float32.
                                         source_file  force_error
0  Li_110_slab__LLZO_010_La_order0_off_bestgap_2....   196.624484
1  Li_110_slab__LLZO_001_Zr_code93_sto_bestgap_2....   119.055250


In [3]:

import numpy as np
import pandas as pd
import torch
from ase.io import read
from mace.calculators import MACECalculator

datasets = [
    "/home/mehuldarak/athena/DFTFE_labelled_data/completedRuns_2.extxyz"
]

model_path = "/home/mehuldarak/athena/mace_seed_1_trial0_compiled.model"

torch.set_default_dtype(torch.float32)

calc = MACECalculator(
    model_path=model_path,
    device="cuda",
    default_dtype="float32"
)

rows = []

for file in datasets:
    structures = read(file, ":")

    for atoms in structures:
        atoms.calc = calc

        pred_forces = atoms.get_forces()
        ref_forces = atoms.arrays["REF_forces"]

        # absolute force error
        diff = pred_forces - ref_forces
        atom_errors = np.linalg.norm(diff, axis=1)

        # one scalar per structure
        force_error = atom_errors.mean()

        rows.append({
            "source_file": atoms.info["filename"],
            "force_error": 1000*force_error
        })

df = pd.DataFrame(rows)

print(df.head())

df.to_csv("maceTr0_vs_runs2_force_errors.csv", index=False)

/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/mace/calculators/mace.py:166: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


Using head Default out of ['Default']
                                         source_file  force_error
0  Li_110_slab__LLZO_010_La_order0_off_bestgap_2....    94.772527
1  Li_110_slab__LLZO_001_Zr_code93_sto_bestgap_2....    98.655872


Energy errors and saving output

In [4]:
from ase.io import read
import pandas as pd
from collections import Counter
import os

# isolated atom energies
E0s = {
    3: -190.7590256408,   # Li
    8: -442.9888796243,   # O
    40: -1380.1817128081, # Zr
    57: -958.0774205521   # La
}

files = [
    "/home/mehuldarak/athena/DFTFE_labelled_data/completedRuns_2.extxyz"
]

rows = []

for f in files:
    atoms_list = read(f, ":")

    for atoms in atoms_list:
        filename = atoms.info["filename"]
        ref_energy = atoms.info["REF_energy"]

        counts = Counter(atoms.get_atomic_numbers())
        e0_sum = sum(E0s[z] * n for z, n in counts.items())

        cohesive_energy = ref_energy - e0_sum

        rows.append({
            "Filename": filename,
            "REF_energy": ref_energy,
            "Cohesive_energy": cohesive_energy
        })

df = pd.DataFrame(rows)
df.to_csv("DFTFE_Runs2_cohesive_energies.csv", index=False)


Cohesive energy extraction from MACE models

In [5]:
import numpy as np
import pandas as pd
import torch
from ase.io import read
from mace.calculators import MACECalculator
from collections import Counter

extxyz_file = "/home/mehuldarak/athena/DFTFE_labelled_data/completedRuns_2.extxyz"

models = {
    "trial0": {
        "path": "/home/mehuldarak/athena/mace_seed_1_trial0_compiled.model",
        "E0s": {
            3: -190.7590332,
            8: -442.9888916,
            40: -1380.181763,
            57: -958.0773926
        },
        "out": "maceTr0_vs_Runs2_cohesive_energies.csv"
    },
    "univ": {
        "path": "/home/mehuldarak/MACE_models/universal_09072025/mace-omat-0-medium.model",
        "E0s": {
            3: -0.29754725,
            8: -1.54838784,
            40: -2.28579303,
            57: -0.45580806
        },
        "out": "maceuni_vs_Runs2_cohesive_energies.csv"
    }
}

torch.set_default_dtype(torch.float32)

structures = read(extxyz_file, ":")

for name, info in models.items():

    calc = MACECalculator(
        model_path=info["path"],
        device="cuda",
        default_dtype="float32"
    )

    rows = []

    for atoms in structures:

        atoms.calc = calc

        pred_energy = atoms.get_potential_energy()

        counts = Counter(atoms.get_atomic_numbers())

        e0_sum = sum(info["E0s"][z] * n for z, n in counts.items())

        cohesive = pred_energy - e0_sum

        rows.append({
            "source_file": atoms.info["filename"],
            "cohesive_energy": cohesive
        })

    df = pd.DataFrame(rows)

    df.to_csv(info["out"], index=False)

    print(f"Saved {info['out']}")

/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/mace/calculators/mace.py:166: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


Using head Default out of ['Default']


Saved maceTr0_vs_Runs2_cohesive_energies.csv
Using head default out of ['default']
Default dtype float32 does not match model dtype float64, converting models to float32.
Saved maceuni_vs_Runs2_cohesive_energies.csv
